# Agilent BioTek PowerWaveX

```{device-card} biotek-powerwave-x
```

| Property | Value |
|---|---|
| Connection | RS-232 serial, 9600 8N2 |
| PLR extra | `serial` |
| Modes | Absorbance: endpoint, kinetic and spectral scan |
| Wavelengths | 200-999 nm, 1 nm increments, up to six per read |
| Plates | 96-well (8x12) and 384-well (16x24) |
| Incubator | Optional, 22-50 degrees C, model dependent |

:::{warning}
Absorbance **data** and **spectral scanning** have not been confirmed against an instrument. Everything else
here has: identification, the wavelength table, temperature, carrier movement, the
assay definition, plate geometry, strip range and the self-test were all exercised
on a PowerWaveX w/ Incubator running firmware R06.20. That instrument's optics fail
their self-test, so no usable readings could be taken from it; the parsing is
verified against serial traffic captured from the same machine in 2021, when its
optics still worked. Please report your results at
[discuss.pylabrobot.org](https://discuss.pylabrobot.org).
:::

The PowerWaveX is an absorbance-only reader from the early 2000s. It predates the
USB BioTek readers and speaks a different command language, so it is a separate
class from {class}`~pylabrobot.agilent.biotek.Cytation5` and
{class}`~pylabrobot.agilent.biotek.SynergyH1` rather than a subclass.

Install the serial extra:

```
pip install pylabrobot[serial]
```

## Communication

The instrument speaks a byte-oriented protocol over RS-232 at 9600 baud, 8 data
bits, no parity and **two** stop bits. Each command is a single ASCII character
answered with `ACK` or `NAK`, usually followed by a five-byte status string.

Wavelengths are selected by a monochromator, so the stored wavelength table is a
*calibration* set rather than a list of installed filters. Reads at other
wavelengths work and simply warn that gains are uncalibrated there.

## Physical setup

Connect a USB-to-RS-232 adapter to the instrument's serial port and power it on.
On Linux the port is typically `/dev/ttyUSB0`; your user needs access to it, which
usually means membership of the `uucp` or `dialout` group. On Windows it is a `COM`
port.

## Create the driver

Pass the serial port, or a USB vendor and product id to have it discovered.

In [ ]:
from pylabrobot.agilent.biotek import PowerWaveX

# A /dev/ttyUSB* path can move between reboots. The by-id symlink does not, so
# prefer `ls /dev/serial/by-id/` and use what you find there.
reader = PowerWaveX(name="reader", port="/dev/ttyUSB0")

## Connect

`setup()` opens the port and identifies the instrument, reading its configuration
word and calibrated wavelength table.

In [ ]:
await reader.setup()

## Identify the instrument

Both values are cached at setup, so these do not talk to the instrument again.

In [ ]:
print(reader.model)
print(reader.wavelength_table)
print(reader.supports_heating)

# -> PowerWaveX w/ Incubator
# -> [550, 620, 665, 520, 450, 595]
# -> True
#
# The wavelength table is the instrument's installed filter set. A read is not
# limited to it -- the monochromator will go anywhere in range -- but gains are
# calibrated for those six, so readings elsewhere are less accurate.

## Move the plate carrier

`open()` presents the carrier so a plate can be loaded, `close()` stores it inside.
Both are idempotent: calling `open()` twice sends nothing the second time.

In [ ]:
await reader.open()
# load a plate
await reader.close()

In [ ]:
await reader.open()


## Incubator

The setpoint is whole degrees, 22 to 50, or 0 to switch the heaters off. The
instrument offers no way to read its setpoint back, and keeps it in non-volatile
memory across power cycles, so a value left by an earlier session is invisible
until the chamber temperature reveals it.

In [ ]:
await reader.set_temperature(30)
print(await reader.request_current_temperature())

# -> 30.0
#
# There is no active cooling: a lower setpoint stops the heaters and the chamber
# then drifts down on its own. `reader.temperature_setpoint` reports what this
# session last wrote, since the instrument cannot be asked.

## Read absorbance

Pass the plate, the wells to read, and a wavelength in nm. The result is a list of
{class}`~pylabrobot.agilent.biotek.plate_reader_base.AbsorbanceResult`, whose
`data` is indexed `[row][column]` in optical density units.

In [ ]:
from pylabrobot.resources import cellvis_96_wellplate_350uL_Fb

plate = cellvis_96_wellplate_350uL_Fb(name="plate")

results = await reader.read_absorbance(plate, plate.get_all_items(), 630)
print(results[0].data[0][0])  # well A1

# -> 0.098
#
# Readings are optical density. An empty light path reads 0.000.

## Read several wavelengths in one pass

The assay definition holds up to six wavelengths and a single pass of the plate
returns one data block for each, so this is far quicker than reading one at a
time.

In [ ]:
results = await reader.read_absorbances(plate, plate.get_all_items(), [450, 630, 990])

for result in results:
    print(result.wavelength, result.data[0][0])

# -> 450 3.218
# -> 630 1.873
# -> 990 0.642
#
# One pass of the plate, so all three share a timestamp. Up to six wavelengths.

## Read part of a plate

The instrument reads whole columns, called strips, and cannot select rows. A
selection that does not fill every row of its columns is widened to the enclosing
strips, and those extra wells are reported rather than discarded. Columns outside
the range were never measured and come back as `None`.

In [ ]:
wells = [w for w in plate.get_all_items() if w.get_column() < 3]

results = await reader.read_absorbance(plate, wells, 630)
print(results[0].data[0][0])   # measured
print(results[0].data[0][11])  # None, never read

# -> 0.098
# -> None
#
# The instrument reads whole columns, so a selection is widened to the columns
# that cover it. Wells outside the range come back as None, not 0.0.

## Kinetic read

The instrument runs the whole series itself, emitting one data block per
wavelength per cycle, so this call blocks for roughly `interval * count` seconds.
Each result carries the timestamp of its cycle.

`request_minimum_kinetic_interval()` reports the shortest interval the loaded assay
allows; a shorter request is accepted but warns, and cycles will run late.

In [ ]:
results = await reader.read_absorbance_kinetic(
    plate,
    plate.get_all_items(),
    wavelengths=[600, 630],
    interval=60,
    count=30,
)

for result in results[:4]:
    print(result.timestamp, result.wavelength, result.data[0][0])

### Choosing a kinetic interval

The shortest interval the instrument will accept depends on the assay, so ask
for it only once that assay is loaded -- which `read_absorbance_kinetic` does
itself. Asking beforehand answers for whatever was loaded last.

In [ ]:
# Measured on a 96-well plate at 9600 baud: 10s for one wavelength, 19s for two,
# 38s for four. An interval below the minimum raises ValueError naming it, since
# the instrument would refuse the read outright.
try:
    await reader.read_absorbance_kinetic(
        plate, plate.get_all_items(), [450, 620], interval=15, count=2
    )
except ValueError as e:
    print(e)

# -> interval 15s is below the instrument's minimum of 19s for this assay. ...
#
# Once above the minimum the interval holds: 12 cycles at 15s drifted 0.38s in
# total.

## Spectral scan

A scan sweeps the monochromator across a range for **one well or one column**,
rather than reading the whole plate at a few fixed wavelengths. Use it to find
an absorbance maximum, or to check a spectrum.

At least two wavelengths must be scanned, so the step cannot exceed the span.

In [ ]:
result = await reader.scan_plate(
    plate,
    start_wavelength=400,
    stop_wavelength=700,
    step=10,
    well=plate.get_item(0),  # A1
)

for wavelength, absorbance in result.spectrum():
    print(wavelength, absorbance)

# -> 400 3.267
# -> 410 3.401
# -> 420 inf      <- saturated at this wavelength
# ...
# -> 690 1.998
# -> 700 1.997
#
# A scan sweeps the monochromator over one well, which is a different operation
# from read_absorbance() reading the whole plate at fixed wavelengths.

### Scanning a whole column

Passing `column` instead of `well` scans every row of that column at once. The
instrument uses two passes internally, because even-row detectors sit one column
to the left; the driver handles that transparently.

`data` is then indexed `[wavelength][row]`.

In [ ]:
result = await reader.scan_plate(
    plate, start_wavelength=400, stop_wavelength=700, step=25, column=1
)

print(result.wavelengths)
print(result.data[0])        # every row at the first wavelength
print(result.spectrum(3))    # the spectrum of row D

### Calibration

`calibrate=True` calibrates every wavelength before scanning, which the manual
recommends when absolute accuracy matters. Without it the instrument still
calibrates any wavelength not scanned since power-up, and any that previously
errored. `result.calibrated` reports what it actually did.

In [ ]:
result = await reader.scan_plate(
    plate,
    start_wavelength=400,
    stop_wavelength=700,
    step=10,
    well=plate.get_item(0),
    calibrate=True,
)
print(result.calibrated)

# -> [400, 410, 420, 430, 440, 450, 460, 470, 480, 490, 500, ...]
#
# `calibrated` lists the wavelengths the instrument had to calibrate for this
# scan. Without calibrate=True it only calibrates what it has not seen since
# power-on, so the list is usually empty on a second scan of the same band.

## Overrange wells

A well that absorbs more than the instrument can measure comes back as `*****`
on the wire, which the driver reports as `math.inf`. That is distinct from
`None`, which means the well was never measured.

In [ ]:
import math

results = await reader.read_absorbance(plate, plate.get_all_items(), 450)
grid = results[0].data

saturated = [(r, c) for r, row in enumerate(grid)
             for c, v in enumerate(row) if v is not None and math.isinf(v)]
print(len(saturated), "saturated wells")
print(grid[1][3])

# -> 30 saturated wells
# -> inf
#
# A warning is logged with the count. Note that "*****" is not simply
# "above 3.000 OD", the top of the specified range: values above it, up to at
# least 4.05, still arrive as numbers. They parse, but do not trust them as
# measurements.

## Reading below 210 nm

The monochromator reaches 200 nm, but a plate read has to set its own gain for
a wavelength it has not calibrated, and below 210 nm there is too little light
for that to succeed. The read is then abandoned and the instrument stops
answering until someone presses the key on its front panel.

A scan calibrates every wavelength it sweeps and manages it even at 200 nm, so
scan the band first and the read works. One scan per power-on is enough.

In [ ]:
# Calibrate 200-210 nm, then read inside that band.
await reader.scan_plate(
    plate, start_wavelength=200, stop_wavelength=210, step=5, well=plate.get_item(0)
)
results = await reader.read_absorbance(plate, plate.get_all_items(), 205)
print(results[0].data[0][0])

# -> 0.004
#
# Without the scan first, that read would abandon itself and latch 0F11
# ("signal out of range") on the display. 200 nm is a hard floor: a request
# below it is refused with status 090 and nothing moves.

## Shaking

A scan can shake the plate first, and a kinetic read shakes between cycles.
`shake_time` is in seconds (0-99) and `shake_speed` is 0 slow, 1 medium, 2 fast,
3 cycling.

In [ ]:
import time

t0 = time.monotonic()
await reader.scan_plate(
    plate,
    start_wavelength=500,
    stop_wavelength=520,
    step=10,
    well=plate.get_item(0),
    shake_time=20,
    shake_speed=2,
)
print(f"{time.monotonic() - t0:.1f}s")

# -> 31.5s
#
# The same scan without shaking takes 11.3s, so the 20 seconds are really spent
# shaking.

## Abort a read

`halt()` stops a read or scan in progress, homes every axis and presents the
carrier outside.

## When a read is abandoned

A read can fault part way through -- a wavelength the optics cannot manage, or a
mechanical problem. The instrument does not report the reason over the serial
port: it shows a code on its front panel and stops answering until that is
acknowledged there. The driver raises `PowerWaveReadAbortedError` rather than
leaving you with a bare timeout.

In [ ]:
from pylabrobot.agilent.biotek.powerwave import PowerWaveReadAbortedError

try:
    await reader.read_absorbance(plate, plate.get_all_items(), 205)
except PowerWaveReadAbortedError as e:
    print(e)
    print("reported code:", e.code)

# -> No data block arrived after 0 of 1 block: the instrument sent nothing for
#    70s. It has halted and is waiting for the fault to be acknowledged at its
#    front panel, and will not answer the serial port until then. Press the
#    panel key, note the code it shows, then call self_test() to clear the
#    latched 080. No power cycle is needed.
# -> reported code: None
#
# `code` is set only when the instrument did send a status; most of the time it
# sends nothing and the panel is the only source. Recovery is always the same:
# press the panel key, then self_test().

## Status errors

Anything the instrument rejects raises `PowerWaveStatusError` carrying its
status code. These are ordinary refusals, not faults, and it stays usable.

In [ ]:
from pylabrobot.agilent.biotek.powerwave import PowerWaveStatusError

try:
    await reader.set_plate_geometry(16, 24)  # a 384-well plate
except PowerWaveStatusError as e:
    print(e.code, "-", e)

# -> 0A0 - PowerWave status 0A0: Error in strip range selection
#
# A 96-well instrument refuses the 384-well geometry this way, so this is how
# you find out which model you have. It reads 96-well plates as usual
# afterwards. Other codes worth knowing: 080 instrument failure (clear it with
# self_test()), 090 a bad assay, scan or table definition, 0B0 an incubator
# setpoint outside 22-50 C.

In [ ]:
await reader.halt()

## Self-test

Run this when a read fails with status `080`, and after changing the wavelength
table, since it recalibrates gains. It takes a couple of minutes and moves every
axis.

The self-test's status string reports only whether the *command* ran: a failing
instrument still answers `000`. The driver therefore takes the verdict from the
report body and raises `PowerWaveSelfTestError` with the decoded error code.

In [ ]:
from pylabrobot.agilent.biotek.powerwave import PowerWaveSelfTestError

try:
    print(await reader.self_test())
except PowerWaveSelfTestError as e:
    print(e.code, e.description)

# The report is the instrument's own text, ending in SYSTEM TEST PASS or FAIL.
# It also clears a latched 080 ("instrument failure"), so run it first if reads
# start failing with that status.

## Disconnect

In [ ]:
await reader.stop()